# This is a python algorithm for the CNN-LSTM , CNN-Bi_LSTM and NeuroSymbolic AI Hybrid deep learning models

In [ ]:
Requirements to install 

pip install tensor flow 

pip install prettytable

In [ ]:
#CNN-LSTM
import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Flatten, Dropout, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_curve, auc
import time
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from tensorflow.keras.utils import plot_model 

# Read the CSV file
df = pd.read_csv('Final_clean_sampl_norm_balance_shuff_NF-UQ-NIDS.csv')

# Separate the features (X) and the target variable (y)
X = df.drop(columns=['Label']).values
y = df['Label'].values

# Get the input shape
input_shape = X.shape
print("Input shape:", input_shape)

# Define the number of folds for cross-validation
n_folds = 5

# Initialize the k-fold cross-validation
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Initialize lists to store the training and validation loss and accuracy histories
train_loss_histories = []
val_loss_histories = []
train_acc_histories = []
val_acc_histories = []

# Perform k-fold cross-validation
for fold, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"Fold {fold+1}/{n_folds}")
    
    # Split the data into training and testing sets for the current fold
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Define the CNN-LSTM model
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1], 1)))
    model.add(Conv1D(filters=32, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(LSTM(64, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(32))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))
    
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    
    # Print the model summary
    print("Model Summary:")
    model.summary(print_fn=lambda x: print(x, end="\n\n"))
    
    # Start the timer
    start_time = time.time()
    
    # Train the model
    history = model.fit(X_train.reshape(X_train.shape[0], X_train.shape[1], 1), y_train,
                       epochs=10, batch_size=32, validation_data=(X_test.reshape(X_test.shape[0], X_test.shape[1], 1), y_test), verbose=1)
    
    # Append the training and validation loss and accuracy histories
    train_loss_histories.append(history.history['loss'])
    val_loss_histories.append(history.history['val_loss'])
    train_acc_histories.append(history.history['accuracy'])
    val_acc_histories.append(history.history['val_accuracy'])
    
    # Calculate the elapsed time
    elapsed_time = time.time() - start_time
    
    # Calculate the time remaining
    total_time = elapsed_time * n_folds / (fold + 1)
    time_remaining = total_time - elapsed_time
    
    print(f"Elapsed time: {elapsed_time:.2f} seconds")
    print(f"Estimated time remaining: {time_remaining:.2f} seconds")
    
    # Evaluate the model on the test set for the current fold
    y_pred = model.predict(X_test.reshape(X_test.shape[0], X_test.shape[1], 1))
    y_pred_binary = (y_pred > 0.5).astype(int)
    
    precision = precision_score(y_test, y_pred_binary)
    recall = recall_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_binary).ravel()
    specificity = tn / (tn + fp)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    
    # Print the confusion matrix using PrettyTable
    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_binary)
    table = PrettyTable()
    table.field_names = ["", "Predicted Negative", "Predicted Positive"]
    table.add_row(["Actual Negative", cm[0,0], cm[0,1]])
    table.add_row(["Actual Positive", cm[1,0], cm[1,1]])
    print(table)
    
    fpr, tpr, _ = roc_curve(y_test, y_pred)
    roc_auc = auc(fpr, tpr)
    print(f"ROC AUC: {roc_auc:.4f}")
    
    test_loss, test_accuracy = model.evaluate(X_test.reshape(X_test.shape[0], X_test.shape[1], 1), y_test, verbose=0)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Plot the training and validation loss histories
plt.figure(figsize=(10, 6))
for i in range(n_folds):
    plt.plot(train_loss_histories[i], label=f'Training Loss (Fold {i+1})')
    plt.plot(val_loss_histories[i], label=f'Validation Loss (Fold {i+1})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig('loss_plot.png')
print('Loss plot saved as loss_plot.png')

# Plot the training and validation accuracy histories
plt.figure(figsize=(10, 6))
for i in range(n_folds):
    plt.plot(train_acc_histories[i], label=f'Training Accuracy (Fold {i+1})')
    plt.plot(val_acc_histories[i], label=f'Validation Accuracy (Fold {i+1})')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)
plt.savefig('accuracy_plot.png')
print('Accuracy plot saved as accuracy_plot.png')

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.savefig('roc_plot.png')
print('ROC plot saved as roc_plot.png')

In [ ]:
# CNN Bi-LSTM MODEL

import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Flatten, Dropout, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_curve, auc
import time
from tensorflow.keras.utils import plot_model
import matplotlib.pyplot as plt
from prettytable import PrettyTable

# Read the CSV file
df = pd.read_csv('Final_clean_sampl_norm_balance_shuff_NF-UQ-NIDS.csv')

# Separate the features (X) and the target variable (y)
X = df.drop(columns=['Label']).values
y = df['Label'].values

# Get the input shape
input_shape = X.shape
print("Input shape:", input_shape)

# Define the number of folds for cross-validation
n_folds = 2

# Initialize the k-fold cross-validation
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Initialize lists to store the training and validation loss and accuracy histories
train_loss_histories = []
val_loss_histories = []
train_acc_histories = []
val_acc_histories = []

# Perform k-fold cross-validation
for fold, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"Fold {fold+1}/{n_folds}")
    
    # Split the data into training and testing sets for the current fold
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Define the CNN-Bi-LSTM model
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1], 1)))
    model.add(Conv1D(filters=32, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Bidirectional(LSTM(64, return_sequences=True, activation='relu')))
    model.add(Dropout(0.2))
    model.add(Bidirectional(LSTM(32, activation='relu')))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))
    
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    
    # Print the model summary
    print("Model Summary:")
    model.summary(print_fn=lambda x: print(x, end="\n\n"))
    
    # Generate the model architecture image
    plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)
    print('Model architecture saved as model_architecture.png')
    
    # Start the timer
    start_time = time.time()
    
    # Train the model
    history = model.fit(X_train.reshape(X_train.shape[0], X_train.shape[1], 1), y_train,
                       epochs=50, batch_size=32, validation_data=(X_test.reshape(X_test.shape[0], X_test.shape[1], 1), y_test), verbose=1)
    
    # Append the training and validation loss and accuracy histories
    train_loss_histories.append(history.history['loss'])
    val_loss_histories.append(history.history['val_loss'])
    train_acc_histories.append(history.history['accuracy'])
    val_acc_histories.append(history.history['val_accuracy'])
    
    # Calculate the elapsed time
    elapsed_time = time.time() - start_time
    
    # Calculate the time remaining
    total_time = elapsed_time * n_folds / (fold + 1)
    time_remaining = total_time - elapsed_time
    
    print(f"Elapsed time: {elapsed_time:.2f} seconds")
    print(f"Estimated time remaining: {time_remaining:.2f} seconds")
    
    # Evaluate the model on the test set for the current fold
    y_pred = model.predict(X_test.reshape(X_test.shape[0], X_test.shape[1], 1))
    y_pred_binary = (y_pred > 0.5).astype(int)
    
    precision = precision_score(y_test, y_pred_binary)
    recall = recall_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_binary).ravel()
    specificity = tn / (tn + fp)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    
    # Print the confusion matrix using PrettyTable
    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_binary)
    table = PrettyTable()
    table.field_names = ["", "Predicted Negative", "Predicted Positive"]
    table.add_row(["Actual Negative", cm[0,0], cm[0,1]])
    table.add_row(["Actual Positive", cm[1,0], cm[1,1]])
    print(table)
    
    fpr, tpr, _ = roc_curve(y_test, y_pred)
    roc_auc = auc(fpr, tpr)
    print(f"ROC AUC: {roc_auc:.4f}")
    
    test_loss, test_accuracy = model.evaluate(X_test.reshape(X_test.shape[0], X_test.shape[1], 1), y_test, verbose=0)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Plot the training and validation loss histories
plt.figure(figsize=(10, 6))
for i in range(n_folds):
    plt.plot(train_loss_histories[i], label=f'Training Loss (Fold {i+1})')
    plt.plot(val_loss_histories[i], label=f'Validation Loss (Fold {i+1})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig('loss_plot.png')
print('Loss plot saved as loss_plot.png')

# Plot the training and validation accuracy histories
plt.figure(figsize=(10, 6))
for i in range(n_folds):
    plt.plot(train_acc_histories[i], label=f'Training Accuracy (Fold {i+1})')
    plt.plot(val_acc_histories[i], label=f'Validation Accuracy (Fold {i+1})')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)
plt.savefig('accuracy_plot.png')
print('Accuracy plot saved as accuracy_plot.png')

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.savefig('roc_plot.png')
print('ROC plot saved as roc_plot.png')

In [ ]:
###Neuro Symbolic AI 

import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Flatten, Dropout, Input, Layer
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_curve, auc
import time
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from tensorflow.keras import backend as K

# Read the CSV file
df = pd.read_csv('Final_PreProcessed_dataset.csv')

# Separate the features (X) and the target variable (y)
X = df.drop(columns=['Label','Attack']).values
y = df['Label'].values

# Get the input shape
input_shape = X.shape
print("Input shape:", input_shape)

# Define the number of folds for cross-validation
n_folds = 2

# Initialize the k-fold cross-validation
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Initialize lists to store the training and validation loss and accuracy histories
train_loss_histories = []
val_loss_histories = []
train_acc_histories = []
val_acc_histories = []

# Custom Attention layer
class Attention(Layer):
    def __init__(self, return_sequences=True, **kwargs):
        self.return_sequences = return_sequences
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1), initializer="zeros")
        super(Attention, self).build(input_shape)

    def call(self, x):
        # Alignment scores
        e = K.tanh(K.dot(x, self.W) + self.b)
        a = K.softmax(e, axis=1)
        
        # Weighted sum
        output = x * a
        
        if self.return_sequences:
            return output
        else:
            return K.sum(output, axis=1)

# Perform k-fold cross-validation
for fold, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"Fold {fold+1}/{n_folds}")
    
    # Split the data into training and testing sets for the current fold
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Define the CNN-LSTM Model with Attention
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1], 1)))
    model.add(Conv1D(filters=32, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(LSTM(64, return_sequences=True))
    model.add(Attention(return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(32))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))
    
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    
    # Print the model summary
    print("Model Summary:")
    model.summary(print_fn=lambda x: print(x, end="\n\n"))
    
    # Generate the model architecture image
    plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)
    print('Model architecture saved as model_architecture.png')
    
    # Start the timer
    start_time = time.time()
    
    # Train the model
    history = model.fit(X_train.reshape(X_train.shape[0], X_train.shape[1], 1), y_train,
                       epochs=2, batch_size=32, validation_data=(X_test.reshape(X_test.shape[0], X_test.shape[1], 1), y_test), verbose=1)
    
    # Append the training and validation loss and accuracy histories
    train_loss_histories.append(history.history['loss'])
    val_loss_histories.append(history.history['val_loss'])
    train_acc_histories.append(history.history['accuracy'])
    val_acc_histories.append(history.history['val_accuracy'])
    
    # Calculate the elapsed time
    elapsed_time = time.time() - start_time
    
    # Calculate the time remaining
    total_time = elapsed_time * n_folds / (fold + 1)
    time_remaining = total_time - elapsed_time
    
    print(f"Elapsed time: {elapsed_time:.2f} seconds")
    print(f"Estimated time remaining: {time_remaining:.2f} seconds")
    
    # Evaluate the model on the test set for the current fold
    y_pred = model.predict(X_test.reshape(X_test.shape[0], X_test.shape[1], 1))
    y_pred_binary = (y_pred > 0.5).astype(int)
    
    precision = precision_score(y_test, y_pred_binary)
    recall = recall_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_binary).ravel()
    specificity = tn / (tn + fp)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    
    # Print the confusion matrix using PrettyTable
    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_binary)
    table = PrettyTable()
    table.field_names = ["", "Predicted Negative", "Predicted Positive"]
    table.add_row(["Actual Negative", cm[0,0], cm[0,1]])
    table.add_row(["Actual Positive", cm[1,0], cm[1,1]])
    print(table)
    
    fpr, tpr, _ = roc_curve(y_test, y_pred)
    roc_auc = auc(fpr, tpr)
    print(f"ROC AUC: {roc_auc:.4f}")
    
    test_loss, test_accuracy = model.evaluate(X_test.reshape(X_test.shape[0], X_test.shape[1], 1), y_test, verbose=0)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Plot the training and validation loss histories
plt.figure(figsize=(10, 6))
for i in range(n_folds):
    plt.plot(train_loss_histories[i], label=f'Training Loss (Fold {i+1})')
    plt.plot(val_loss_histories[i], label=f'Validation Loss (Fold {i+1})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig('loss_plot.png')
print('Loss plot saved as loss_plot.png')

# Plot the training and validation accuracy histories
plt.figure(figsize=(10, 6))
for i in range(n_folds):
    plt.plot(train_acc_histories[i], label=f'Training Accuracy (Fold {i+1})')
    plt.plot(val_acc_histories[i], label=f'Validation Accuracy (Fold {i+1})')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)
plt.savefig('accuracy_plot.png')
print('Accuracy plot saved as accuracy_plot.png')

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.savefig('roc_plot.png')
print('ROC plot saved as roc_plot.png')